# 00 -- Data setup and exploration
Audits the G2F 2024/2025 GxE Prediction Competition training data: confirms every
expected file actually contains data (not a failed/HTML download), profiles each
file's shape and columns, and cross-checks environment (year x location) codes
across the trait, meta, soil, weather, and EC files so join keys are understood
before any effect-alone model is built.

No modeling in this notebook. Execution only -- if this needs to run again
after real column names are confirmed against `readme.txt`, extend the checks
below rather than rewriting them.

## Environment setup (Colab or local)

In [39]:
from pathlib import Path
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/g2f_effect_decomposition')
else:
    # Local run (VSCode). Data and results live on Drive; point this at wherever
    # Drive is mounted -- e.g. Google Drive for Desktop on WSL2 is typically
    # under /mnt/g/My Drive/... (adjust drive letter as needed).
    # G2F_BASE_PATH env var overrides this for testing without editing the notebook.
    BASE_PATH = Path(os.environ.get('G2F_BASE_PATH', '/mnt/g/My Drive/g2f_effect_decomposition'))

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab | BASE_PATH = /content/drive/MyDrive/g2f_effect_decomposition


## Imports and config

In [40]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

DATA_DIR = BASE_PATH / 'data' / 'raw' / 'Training_data'

# Filenames as released in the CyVerse Training_data folder.
EXPECTED_FILES = {
    'trait':            '1_Training_Trait_Data_2014_2023.csv',
    'meta':             '2_Training_Meta_Data_2014_2023.csv',
    'soil':             '3_Training_Soil_Data_2015_2023.csv',
    'weather_full':     '4_Training_Weather_Data_2014_2023_full_year.csv',
    'weather_seasons':  '4_Training_Weather_Data_2014_2023_seasons_only.csv',
    'genotype_vcf':     '5_Genotype_Data_All_2014_2025_Hybrids.vcf',
    'genotype_numeric': '5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt',
    'ec':               '6_Training_EC_Data_2014_2023.csv',
    'key_inbreds':      'key_inbreds_G2F_2014-2025.txt',
}

pd.set_option('display.max_columns', 50)

## File integrity check
CyVerse gates downloads behind a reCAPTCHA click-through in its browser app.
A file pulled with a plain HTTP client (rather than a real browser download)
can silently come back as the ~7KB Angular landing page instead of the actual
data. Every file gets checked for that before anything downstream trusts it.

In [41]:
def integrity_status(path: Path) -> str:
    """Returns 'missing', 'suspect_html', or 'ok' for a downloaded file."""
    if not path.exists():
        return 'missing'
    with open(path, 'rb') as f:
        head = f.read(512)
    if b'<!DOCTYPE html' in head or b'<html' in head[:200]:
        return 'suspect_html'
    return 'ok'


status_rows = []
for key, filename in EXPECTED_FILES.items():
    path = DATA_DIR / filename
    status = integrity_status(path)
    size = path.stat().st_size if path.exists() else None
    status_rows.append({'key': key, 'file': filename, 'status': status, 'size_bytes': size})

status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))

bad = status_df[status_df['status'] != 'ok']
if len(bad):
    print(f"\n{len(bad)} file(s) failed the integrity check -- re-download these before"
          " trusting anything profiled below:")
    print(bad['file'].to_string(index=False))
else:
    print("\nAll files passed the integrity check.")

             key                                                file status  size_bytes
           trait                 1_Training_Trait_Data_2014_2023.csv     ok    31590253
            meta                  2_Training_Meta_Data_2014_2023.csv     ok      100054
            soil                  3_Training_Soil_Data_2015_2023.csv     ok       28101
    weather_full     4_Training_Weather_Data_2014_2023_full_year.csv     ok    10460386
 weather_seasons  4_Training_Weather_Data_2014_2023_seasons_only.csv     ok     5493581
    genotype_vcf           5_Genotype_Data_All_2014_2025_Hybrids.vcf     ok    57432657
genotype_numeric 5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt     ok    40765753
              ec                    6_Training_EC_Data_2014_2023.csv     ok     1971244
     key_inbreds                       key_inbreds_G2F_2014-2025.txt     ok      230060

All files passed the integrity check.


## Tabular file profiles
Loads a small sample of each CSV that passed the integrity check: shape,
columns, dtypes, and a head preview. Skips any file that failed the check
above rather than profiling garbage.

In [ ]:
TABULAR_KEYS = ['trait', 'meta', 'soil', 'weather_full', 'weather_seasons', 'ec']

tabular_frames: dict[str, pd.DataFrame] = {}

for key in TABULAR_KEYS:
    filename = EXPECTED_FILES[key]
    row = status_df[status_df['key'] == key].iloc[0]
    if row['status'] != 'ok':
        print(f"[skip] {key} ({filename}): {row['status']}")
        continue

    df_full = pd.read_csv(DATA_DIR / filename)
    tabular_frames[key] = df_full

    print(f"=== {key} ({filename}) ===")
    print(f"shape: {df_full.shape}")
    print(f"columns: {list(df_full.columns)}")
    print(df_full.head(3))
    print()

## Missing-value profile
Per column missing-value percentage for each loaded tabular file -- flags
which fields are usable as-is vs. need imputation or exclusion.

In [ ]:
for key, df_full in tabular_frames.items():
    miss = (df_full.isna().mean() * 100).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"=== {key}: missing % by column ===")
    print(miss.to_string() if len(miss) else "(no missing values)")
    print()

## Environment (year x location) join-key audit
G2F's known misjoin risk is in the environment code used to link trait,
meta, soil, weather, and EC records. This looks for any column whose name
contains 'env' (case-insensitive) in each loaded file, then compares the
sets of values across files -- mismatches here mean the join key isn't as
simple as a direct string match and the readme needs to be checked before
building any loader.

In [ ]:
env_value_sets: dict[str, set] = {}

for key, df_full in tabular_frames.items():
    env_cols = [c for c in df_full.columns if 'env' in c.lower()]
    if not env_cols:
        print(f"{key}: no column with 'env' in its name -- columns are {list(df_full.columns)}")
        continue
    col = env_cols[0]
    env_value_sets[key] = set(df_full[col].astype(str).unique())
    print(f"{key}: using column '{col}' ({df_full[col].nunique()} unique values)"
          + (f" -- also found {env_cols[1:]}" if len(env_cols) > 1 else ""))

print()
keys = list(env_value_sets.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a, b = keys[i], keys[j]
        overlap = env_value_sets[a] & env_value_sets[b]
        only_a = env_value_sets[a] - env_value_sets[b]
        only_b = env_value_sets[b] - env_value_sets[a]
        print(f"{a} vs {b}: {len(overlap)} shared, {len(only_a)} only in {a}, {len(only_b)} only in {b}")
        if only_a:
            print(f"  sample only in {a}: {sorted(only_a)[:5]}")
        if only_b:
            print(f"  sample only in {b}: {sorted(only_b)[:5]}")

## Genotype data
Profiles both genotype formats without loading the full VCF body into
memory -- header/sample/contig info only via `cyvcf2`, plus a shape check
on the numerical marker matrix and a preview of the key-inbreds list.

In [ ]:
geno_vcf_status = status_df[status_df['key'] == 'genotype_vcf'].iloc[0]['status']

if geno_vcf_status == 'ok':
    from cyvcf2 import VCF

    vcf_path = DATA_DIR / EXPECTED_FILES['genotype_vcf']
    vcf = VCF(str(vcf_path))
    print(f"VCF samples: {len(vcf.samples)}")
    print(f"VCF sample preview: {vcf.samples[:5]}")

    n_variants = 0
    contigs = set()
    for i, variant in enumerate(vcf):
        contigs.add(variant.CHROM)
        n_variants += 1
        if i >= 9999:  # cap the scan -- full-file variant count can be done separately if needed
            print("(stopped after 10,000 variants -- re-run without the cap for an exact count)")
            break
    print(f"variants scanned: {n_variants}")
    print(f"contigs seen: {sorted(contigs)}")
else:
    print(f"[skip] genotype_vcf: {geno_vcf_status}")

In [ ]:
geno_num_status = status_df[status_df['key'] == 'genotype_numeric'].iloc[0]['status']

if geno_num_status == 'ok':
    geno_num_path = DATA_DIR / EXPECTED_FILES['genotype_numeric']

    # First line is a format tag (e.g. '<Numeric>'), not part of the
    # tab-separated header -- sep=None sniffing gets fooled by it since the
    # tag line itself has no delimiter to detect. Skip it and read explicitly.
    with open(geno_num_path) as f:
        tag_line = f.readline().strip()
    print(f"format tag line: {tag_line!r}")

    geno_num = pd.read_csv(geno_num_path, sep='\t', skiprows=1, index_col=0)
    geno_num.index.name = 'Hybrid'
    print(f"numerical genotype matrix shape: {geno_num.shape}  (hybrids x markers)")
    print(f"hybrid preview: {list(geno_num.index[:5])}")
    print(f"marker preview: {list(geno_num.columns[:5])}")
    print(geno_num.iloc[:5, :5])
    print()
    print(f"value counts (first marker column): {geno_num.iloc[:, 0].value_counts(dropna=False).to_dict()}")
else:
    print(f"[skip] genotype_numeric: {geno_num_status}")

In [ ]:
key_inbreds_status = status_df[status_df['key'] == 'key_inbreds'].iloc[0]['status']

if key_inbreds_status == 'ok':
    key_inbreds_path = DATA_DIR / EXPECTED_FILES['key_inbreds']
    key_inbreds = pd.read_csv(key_inbreds_path, sep='\t')
    print(f"key_inbreds shape: {key_inbreds.shape}")
    print(f"columns: {list(key_inbreds.columns)}")
    print(key_inbreds.head(10))
    print()
    print(f"Dataset value counts: {key_inbreds['Dataset'].value_counts().to_dict()}")
else:
    print(f"[skip] key_inbreds: {key_inbreds_status}")

## Testing data (2024 held-out set)
Same checks as training, applied to `Testing_data/`. No genotype file here --
the training genotype VCF/numerical matrix already covers 2014-2025 hybrids
and is shared across both splits. Two things specific to the test set matter
more than the individual file profiles: whether test environments are truly
disjoint from training environments (confirms this is a genuine year-based
holdout, not something that needs a custom split), and whether every hybrid
we need to predict actually has genotype coverage.

In [42]:
TEST_DATA_DIR = BASE_PATH / 'data' / 'raw' / 'Testing_data'

EXPECTED_TEST_FILES = {
    'submission_template': '1_Submission_Template_2024.csv',
    'test_meta':           '2_Testing_Meta_Data_2024.csv',
    'test_soil':           '3_Testing_Soil_Data_2024.csv',
    'test_weather_full':   '4_Testing_Weather_Data_2024_full_year.csv',
    'test_weather_seasons':'4_Testing_Weather_Data_2024_seasons_only.csv',
    'test_ec':             '6_Testing_EC_Data_2024.csv',
    'test_observed':       '7_Testing_Observed_Values.csv',
}

test_status_rows = []
for key, filename in EXPECTED_TEST_FILES.items():
    path = TEST_DATA_DIR / filename
    status = integrity_status(path)
    size = path.stat().st_size if path.exists() else None
    test_status_rows.append({'key': key, 'file': filename, 'status': status, 'size_bytes': size})

test_status_df = pd.DataFrame(test_status_rows)
print(test_status_df.to_string(index=False))

bad_test = test_status_df[test_status_df['status'] != 'ok']
if len(bad_test):
    print(f"\n{len(bad_test)} test file(s) failed the integrity check.")
else:
    print("\nAll test files passed the integrity check.")

                 key                                         file status  size_bytes
 submission_template               1_Submission_Template_2024.csv     ok      336283
           test_meta                 2_Testing_Meta_Data_2024.csv     ok       10888
           test_soil                 3_Testing_Soil_Data_2024.csv     ok        3697
   test_weather_full    4_Testing_Weather_Data_2024_full_year.csv     ok      702794
test_weather_seasons 4_Testing_Weather_Data_2024_seasons_only.csv     ok      424779
             test_ec                   6_Testing_EC_Data_2024.csv     ok      189448
       test_observed                7_Testing_Observed_Values.csv     ok      420463

All test files passed the integrity check.


In [43]:
test_tabular_frames: dict[str, pd.DataFrame] = {}

for key, filename in EXPECTED_TEST_FILES.items():
    row = test_status_df[test_status_df['key'] == key].iloc[0]
    if row['status'] != 'ok':
        print(f"[skip] {key} ({filename}): {row['status']}")
        continue

    df_full = pd.read_csv(TEST_DATA_DIR / filename)
    test_tabular_frames[key] = df_full

    print(f"=== {key} ({filename}) ===")
    print(f"shape: {df_full.shape}")
    print(f"columns: {list(df_full.columns)}")
    print(df_full.head(3))
    print()

=== submission_template (1_Submission_Template_2024.csv) ===
shape: (10057, 3)
columns: ['Env', 'Hybrid', 'Yield_Mg_ha']
         Env        Hybrid  Yield_Mg_ha
0  DEH1_2024  01CSI6/LH287          NaN
1  DEH1_2024  01DIB2/LH287          NaN
2  DEH1_2024  01DIB2/PHP02          NaN

=== test_meta (2_Testing_Meta_Data_2024.csv) ===
shape: (23, 40)
columns: ['Year', 'Env', 'Experiment_Code', 'Treatment', 'City', 'Farm', 'Field', 'Trial_ID (Assigned by collaborator for internal reference)', 'Soil_Taxonomic_ID and horizon description, if known', 'Weather_Station_Serial_Number (Last four digits, e.g. m2700s#####)', 'Weather_Station_Latitude (in decimal numbers NOT DMS)', 'Weather_Station_Longitude (in decimal numbers NOT DMS)', 'Date_weather_station_placed', 'Date_weather_station_removed', 'Previous_Crop', 'Pre-plant_tillage_method(s)', 'In-season_tillage_method(s)', 'Type_of_planter (fluted cone; belt cone; air planter)', 'System_Determining_Moisture', 'Pounds_Needed_Soil_Moisture', 'Latitud

In [44]:
for key, df_full in test_tabular_frames.items():
    miss = (df_full.isna().mean() * 100).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"=== {key}: missing % by column ===")
    print(miss.to_string() if len(miss) else "(no missing values)")
    print()

=== submission_template: missing % by column ===
Yield_Mg_ha    100.0

=== test_meta: missing % by column ===
Issue/comment_#5                                                      100.0
Issue/comment_#7                                                      100.0
Issue/comment_#8                                                      100.0
Issue/comment_#6                                                      100.0
Issue/comment_#3                                                       91.3
Issue/comment_#4                                                       91.3
In-season_tillage_method(s)                                            73.9
Issue/comment_#2                                                       73.9
Soil_Taxonomic_ID and horizon description, if known                    65.2
Issue/comment_#1                                                       47.8
Date_weather_station_removed                                           39.1
Field                                                 

### Train/test environment disjointness
If this is a genuine year-based holdout, test environments should not
appear in the training environment sets at all -- any overlap means 2024
environments were also tested in earlier years, which changes what
"held-out" actually means here.

In [45]:
test_env_value_sets: dict[str, set] = {}

for key, df_full in test_tabular_frames.items():
    env_cols = [c for c in df_full.columns if 'env' in c.lower()]
    if not env_cols:
        print(f"{key}: no column with 'env' in its name -- columns are {list(df_full.columns)}")
        continue
    col = env_cols[0]
    test_env_value_sets[key] = set(df_full[col].astype(str).unique())
    print(f"{key}: using column '{col}' ({df_full[col].nunique()} unique values)")

print()
all_test_envs = set().union(*test_env_value_sets.values()) if test_env_value_sets else set()
all_train_envs = set().union(*env_value_sets.values()) if env_value_sets else set()
overlap = all_test_envs & all_train_envs
print(f"Test envs: {len(all_test_envs)} | Train envs: {len(all_train_envs)} | Overlap: {len(overlap)}")
if overlap:
    print(f"  OVERLAPPING envs (investigate before treating this as a clean holdout): {sorted(overlap)[:10]}")
else:
    print("  No overlap -- test environments are a genuine year-based holdout.")

submission_template: using column 'Env' (23 unique values)
test_meta: using column 'Env' (23 unique values)
test_soil: using column 'Env' (16 unique values)
test_weather_full: using column 'Env' (23 unique values)
test_weather_seasons: using column 'Env' (23 unique values)
test_ec: using column 'Env' (22 unique values)
test_observed: using column 'Env' (22 unique values)

Test envs: 23 | Train envs: 272 | Overlap: 0
  No overlap -- test environments are a genuine year-based holdout.


### Hybrid coverage check
Every hybrid in the submission template / observed values needs a row in
the genotype matrix for a genotype-alone model to produce a vote for it.

In [46]:
hybrid_check_key = 'test_observed' if 'test_observed' in test_tabular_frames else 'submission_template'

if hybrid_check_key in test_tabular_frames and 'geno_num' in dir():
    df_check = test_tabular_frames[hybrid_check_key]
    hybrid_cols = [c for c in df_check.columns if 'hybrid' in c.lower()]
    if hybrid_cols:
        col = hybrid_cols[0]
        test_hybrids = set(df_check[col].astype(str).unique())
        genotyped_hybrids = set(geno_num.index.astype(str))
        missing = test_hybrids - genotyped_hybrids
        print(f"{hybrid_check_key}: {len(test_hybrids)} unique hybrids (column '{col}')")
        print(f"Genotyped hybrids available: {len(genotyped_hybrids)}")
        print(f"Test hybrids WITHOUT genotype coverage: {len(missing)}")
        if missing:
            print(f"  sample missing: {sorted(missing)[:10]}")
    else:
        print(f"{hybrid_check_key}: no column with 'hybrid' in its name -- columns are {list(df_check.columns)}")
else:
    print("Skipped -- either the test frame or the genotype matrix (geno_num) isn't available above.")

test_observed: 1063 unique hybrids (column 'Hybrid')
Genotyped hybrids available: 5899
Test hybrids WITHOUT genotype coverage: 0


## Summary
Consolidated status -- what's confirmed usable vs. what still needs a
re-download or a readme check, so the next session knows exactly where to
pick up.

In [47]:
print("=== Training ===")
print(status_df.to_string(index=False))
print()
print("=== Testing ===")
print(test_status_df.to_string(index=False))
print()
print(f"Training tabular files loaded: {list(tabular_frames.keys())}")
print(f"Testing tabular files loaded: {list(test_tabular_frames.keys())}")
print(f"Env join-key sets compared (train): {list(env_value_sets.keys())}")
print(f"Env join-key sets compared (test): {list(test_env_value_sets.keys())}")

=== Training ===
             key                                                file status  size_bytes
           trait                 1_Training_Trait_Data_2014_2023.csv     ok    31590253
            meta                  2_Training_Meta_Data_2014_2023.csv     ok      100054
            soil                  3_Training_Soil_Data_2015_2023.csv     ok       28101
    weather_full     4_Training_Weather_Data_2014_2023_full_year.csv     ok    10460386
 weather_seasons  4_Training_Weather_Data_2014_2023_seasons_only.csv     ok     5493581
    genotype_vcf           5_Genotype_Data_All_2014_2025_Hybrids.vcf     ok    57432657
genotype_numeric 5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt     ok    40765753
              ec                    6_Training_EC_Data_2014_2023.csv     ok     1971244
     key_inbreds                       key_inbreds_G2F_2014-2025.txt     ok      230060

=== Testing ===
                 key                                         file status  size_bytes
 